# PiToMe vs TDA-PiToMe (FloodComplex) Comparison

This notebook compares the original PiToMe energy-based scoring with TDA-PiToMe's GPU-accelerated FloodComplex persistent homology approach.

In [ ]:
# Clone repository
!git clone https://github.com/a11to1n3/PiToMe.git
%cd PiToMe

In [ ]:
import torch
import torch.nn.functional as F
import time
import math
import sys
sys.path.insert(0, '.')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Load TDA module
import importlib.util
spec = importlib.util.spec_from_file_location('tda', './algo/tda-pitome/tda.py')
tda = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tda)

# Load PiToMe
from algo.pitome.merge import pitome_vision

print('✓ Modules loaded')

In [ ]:
# Test configuration
B, T, C = 4, 196, 768  # Typical DeiT: 4 batches, 196 tokens (14x14), 768 dims
RATIO = 0.7  # Keep 70% of tokens

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)

# Simulated token embeddings
x = torch.randn(B, T, C, device=device)
print(f'Input: {x.shape} on {device}')

## PiToMe (Energy-Based Scoring)

In [ ]:
# PiToMe scoring
x_cpu = x.cpu()

t0 = time.time()
merge_pitome = pitome_vision(metric=x_cpu, ratio=RATIO, class_token=False)
merged = merge_pitome(x_cpu.clone())
pitome_time = (time.time() - t0) * 1000

print(f'PiToMe time: {pitome_time:.2f}ms')
print(f'Output: {merged.shape}')

## TDA-PiToMe (FloodComplex GPU)

In [ ]:
# FloodComplex scoring
scorer = tda.FloodComplexScorer(
    landmark_fraction=0.1,
    max_filtration_value=2.0,
    n_filtration_steps=50
)

# Warmup
_ = scorer.compute_scores(x[:1])

# Timed run
if device == 'cuda':
    torch.cuda.synchronize()
t0 = time.time()
scores = scorer.compute_scores(x)
if device == 'cuda':
    torch.cuda.synchronize()
flood_time = (time.time() - t0) * 1000

print(f'FloodComplex time: {flood_time:.2f}ms')
print(f'Scores: {scores.shape}, range [{scores.min():.3f}, {scores.max():.3f}]')

## Comparison

In [ ]:
# Compute PiToMe energy scores for correlation
metric = F.normalize(x_cpu, p=2, dim=-1)
sim = metric @ metric.transpose(-1, -2)
energy = F.elu(sim - 0.5, alpha=1.0).mean(dim=-1)
energy = (energy - energy.min()) / (energy.max() - energy.min() + 1e-8)

# Correlation between methods
flood_cpu = scores.cpu()
corr = torch.corrcoef(torch.stack([flood_cpu[0], energy[0]]))[0, 1]

print('='*50)
print('COMPARISON SUMMARY')
print('='*50)
print(f'{"Method":<20} {"Time (ms)":<15} {"Device"}')
print(f'{"PiToMe":<20} {pitome_time:<15.2f} CPU')
print(f'{"TDA-PiToMe":<20} {flood_time:<15.2f} {device.upper()}')
print(f'\nSpeedup: {pitome_time/flood_time:.2f}x')
print(f'Score correlation: {corr:.3f}')

In [ ]:
# Visualize score distributions
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(energy[0].numpy(), bins=30, alpha=0.7, label='PiToMe Energy')
axes[0].hist(flood_cpu[0].numpy(), bins=30, alpha=0.7, label='FloodComplex')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].set_title('Score Distributions')

axes[1].scatter(energy[0].numpy(), flood_cpu[0].numpy(), alpha=0.5, s=10)
axes[1].set_xlabel('PiToMe Energy')
axes[1].set_ylabel('FloodComplex Score')
axes[1].set_title(f'Correlation: {corr:.3f}')

plt.tight_layout()
plt.show()